# DhakaFlow — Urban Real-Trajectory Pipeline (Colab)
Real pNEUMA vehicles (Athens) → real OSM roads → map-matching → same-site SUMO
twin → published cycle-level calibration (Tsioutis et al., SUMO UC 2026).
Every stage uses official data/tools; repo pins in `upstream/UPSTREAM_PINS.yaml`.
Runtime → Run all (CPU is fine).

In [ ]:
#@title 1 · Setup (~2 min)
%pip -q install eclipse-sumo sumolib traci libsumo leuvenmapmatching nevergrad pandas matplotlib
import os, sumolib; os.environ['SUMO_HOME']=os.path.dirname(sumolib.__file__)+'/../sumo'
print('SUMO ready')

In [ ]:
#@title 2 · Real pNEUMA data (official endpoint, ~230 MB)
import subprocess, hashlib, pathlib
url='https://open-traffic.epfl.ch/wp-content/uploads/mydownloads.php'
subprocess.run(['curl','-s','-X','POST',url,'-A','Mozilla/5.0',
  '--data','fpLocation=d8&fpDate=20181029&fpTime=0800_0830','-o','d8.csv'],check=True)
print('sha256', hashlib.sha256(open('d8.csv','rb').read()).hexdigest()[:16],
      '| bytes', pathlib.Path('d8.csv').stat().st_size)

In [ ]:
#@title 3 · Parse REAL vehicles (provider = research/urban/pneuma_provider.py)
from dataclasses import dataclass
@dataclass
class S: t:float; lat:float; lon:float; kmh:float
def parse(path, hz=1.0):
    step=max(int(round(25.0/hz)),1)*6
    with open(path) as f:
        next(f)
        for line in f:
            p=[x.strip() for x in line.strip().strip(';').split(';')]
            if len(p)<10: continue
            vid,typ=int(p[0]),p[1]; flat=p[4:]; ss=[]
            for j in range(0,len(flat)-5,step):
                try: ss.append(S(float(flat[j+5]),float(flat[j]),float(flat[j+1]),float(flat[j+2])))
                except ValueError: break
            if ss: yield vid,typ,ss
tracks=list(parse('d8.csv'))
print(len(tracks),'REAL vehicles · e.g.',tracks[0][0],tracks[0][1],tracks[0][2][0])

In [ ]:
#@title 4 · Real roads (OSM) + SUMO net of the SAME site
import subprocess
subprocess.run(['curl','-s','-o','athens.osm',
 'https://overpass-api.de/api/map?bbox=23.7282,37.9880,23.7385,37.9945'],check=True)
subprocess.run(['netconvert','--osm-files','athens.osm','-o','athens.net.xml',
 '--keep-edges.by-vclass','passenger','--no-turnarounds','true'],check=True)
import sumolib; net=sumolib.net.readNet('athens.net.xml')
print('edges',len(net.getEdges()),'TLS',len(net.getTrafficLights()))

In [ ]:
#@title 5 · Map-match a few real vehicles (upstream: leuvenmapmatching)
# (full pipeline: scripts/gate_urban_site.py · scripts/gate_pneuma_twin.py)
from leuvenmapmatching.map.inmem import InMemMap
from leuvenmapmatching.matcher.distance import DistanceMatcher
import xml.etree.ElementTree as ET
root=ET.parse('athens.osm').getroot()
nodes={n.get('id'):(float(n.get('lat')),float(n.get('lon'))) for n in root.iter('node')}
DRIV={'primary','secondary','tertiary','residential','unclassified','living_street'}
m=InMemMap('a',use_latlon=True,index_edges=True); used=set()
for w in root.iter('way'):
    tags={t.get('k'):t.get('v') for t in w.iter('tag')}
    if tags.get('highway') not in DRIV: continue
    nds=[nd.get('ref') for nd in w.iter('nd') if nd.get('ref') in nodes]
    for nid in nds:
        if nid not in used: m.add_node(nid,nodes[nid]); used.add(nid)
    ow=tags.get('oneway') in ('yes','true','1')
    for a,b in zip(nds,nds[1:]):
        m.add_edge(a,b)
        if not ow: m.add_edge(b,a)
m.purge()
ok=0
for vid,typ,ss in tracks[:40]:
    pts=[(s.lat,s.lon) for s in ss if 300<=s.t<=420]
    if len(pts)<12: continue
    try: st,_=DistanceMatcher(m,max_dist=45,obs_noise=8,non_emitting_states=True).match(pts)
    except Exception: continue
    if st and len(st)>=len(pts)*0.5: ok+=1
print('matched', ok, 'real vehicles onto real Athens segments')

## 6 · The published calibration (main experiment)
The twin is calibrated with the **authors' own code** of
*Tsioutis, Pourgourides & Timotheou (SUMO UC 2026)* —
`github.com/TsioutisCh/phase-relative-sumo-calibration` @9e25290 —
NGOpt over cumulative phase-relative exit-count curves, per-cycle
reinitialization from UAV states. In the repo:
`scripts/gate_pneuma_twin.py` (twin, A-only demand, B validation) and the
reproduction workspace. Gate evidence: `artifacts/gates/` + docs.